In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "postgres")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
conn = create_engine(connection_string)

## Overall test / assertion / generalization exclusions

In [ ]:
import pandas as pd

query = f"SELECT * FROM mv_exclusions_all"
df = pd.read_sql_query(query, conn)
df

## Filtering-based test / assertion / generalization exclusions

In [ ]:
import pandas as pd
import re

# Query and DataFrame creation
query = "SELECT * FROM mv_exclusions_filtering WHERE reject > 0"
df = pd.read_sql_query(query, conn)

# Map 'level' to 'Filter Type'
level_map = {
    '1-TEST': 'Test',
    '2-ASSERTION': 'Assertion',
    '3-GENERALIZATION': 'Generalization'
}
df['Filter Type'] = df['level'].map(level_map)

# Format 'variant' for LaTeX subscripts
def format_variant(variant):
    return re.sub(r'_(\d+)_TRIES$', r'$_{\1}$', variant)

df['Variant'] = df['variant'].apply(format_variant)

# Reorder and rename columns
df = df[['Variant', 'Filter Type', 'filter_name', 'total', 'accept', 'reject', 'defer']]
df.columns = ['Variant', 'Filter Type', 'Filter Name', 'Total', 'Accept', 'Reject', 'Defer']

display(df)

# Build LaTeX table manually
lines = []
lines.append(r"\begin{table}[H]")
lines.append(r"  \caption{Filtering results for tests, assertions, and generalizations by filter and (generalization) variant.}")
lines.append(r"  \label{tab:exclusions-filtering}")
lines.append(r"  \begin{tabular}{lllrrrr}")
lines.append(r"    \toprule")
lines.append(r"    Variant & Filter Type & Filter Name & Total & Accept & Reject & Defer \\")
lines.append(r"    \midrule")

prev_type = df.iloc[0]['Filter Type']
for i, row in df.iterrows():
    # Insert \midrule when Filter Type changes (but not before the first group)
    if i > 0 and row['Filter Type'] != prev_type:
        lines.append(r"    \midrule")
    prev_type = row['Filter Type']
    # Format row
    line = "    {} & {} & {} & {} & {} & {} & {} \\\\".format(
        row['Variant'],
        row['Filter Type'],
        row['Filter Name'],
        int(row['Total']),
        int(row['Accept']),
        int(row['Reject']),
        int(row['Defer']),
    )
    lines.append(line)
lines.append(r"    \bottomrule")
lines.append(r"  \end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)


## Exclusions caused by JPF execution failures

In [ ]:
import pandas as pd

query = f"SELECT * FROM mv_exclusions_jpf"
df = pd.read_sql_query(query, conn)
df

## Exclusions caused by test failures

In [ ]:
import re

query = f"SELECT * FROM mv_exclusions_test_fails"
df = pd.read_sql_query(query, conn)

# Pivot as before
ordered_variants = (
    df[['variant', 'variant_order']]
    .drop_duplicates()
    .sort_values('variant_order')
    ['variant']
    .tolist()
)
pivoted_df = df.pivot(index='failure_type', columns='variant', values='count')
pivoted_df = pivoted_df[ordered_variants].fillna(0).astype(int)

display(pivoted_df)

# --- AUTOMATED HEADER GENERATION ---

# Extract base variant and tries
variant_info = []
for v in pivoted_df.columns:
    m = re.match(r'([A-Z]+)(?:_(\d+)_TRIES)?', v)
    if m:
        base = m.group(1)
        tries = m.group(2) if m.group(2) else '-'
        variant_info.append((v, base, tries))
    else:
        variant_info.append((v, v, '-'))

# Group columns by base variant
from collections import OrderedDict
grouped = OrderedDict()
for v, base, tries in variant_info:
    grouped.setdefault(base, []).append((v, tries))

# Build header rows
header1 = ['Variant']
header2 = ['Tries']
cmidrules = []
col_idx = 2  # LaTeX columns start at 1, first is 'Variant'

for base, cols in grouped.items():
    n = len(cols)
    if n == 1:
        header1.append(base)
        header2.append('-')
        # No cmidrule needed for single columns
        col_idx += 1
    else:
        header1 += [f'\\multicolumn{{{n}}}{{c}}{{{base}}}']
        header2 += [tries for _, tries in cols]
        # cmidrule for this group
        start = col_idx
        end = col_idx + n - 1
        cmidrules.append(f'\\cmidrule(lr){{{start}-{end}}}')
        col_idx += n

header1_line = ' & '.join(header1) + r' \\'
header2_line = ' & '.join(header2) + r' \\'
cmidrules_line = '\n    '.join(cmidrules)

# --- BUILD THE TABLE ---
latex_table = r"""\begin{table}[H]
  \caption{Number of test execution failures by exception type and (generalization) variant.}
  \label{tab:exclusions-test-fails}
  \begin{tabular}{l""" + "r" * (len(pivoted_df.columns)) + r"""}
    \toprule
    """ + header1_line + "\n    " + cmidrules_line + "\n    " + header2_line + r"""
    \midrule
"""

# Data rows
for failure_type, row in pivoted_df.iterrows():
    row_str = "    " + failure_type + " & " + " & ".join(str(x) for x in row.values) + r" \\"
    latex_table += row_str + "\n"

latex_table += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

print(latex_table)